# Detection and pitch stages on a GPU (Kaggle or Colab)

Runs `detection` and `pitch` for one or more clips and packs `data/processed/<clip_id>/` of every
clip into one zip (`detections.parquet`, `positions.parquet`, `homographies.parquet`, summaries
and debug videos). The events stage is cheap and runs locally afterwards.

1. Enable a GPU (Kaggle: Settings -> Accelerator -> GPU; Colab: Runtime -> Change runtime type)
   and internet access (needed for git clone, packages and model weights).
2. Attach the clips: on Kaggle add your dataset with the `.mp4` files as input; on Colab put
   them on Google Drive.
3. Set the parameters below and run all cells. Download the zip from the output panel.

In [ ]:
# Parameters
REPO_URL = "https://github.com/lendvaioli/taktikai.git"
VIDEO_DIR = ""  # e.g. /kaggle/input/taktikai-dfl-clips or /content/drive/MyDrive/dfl
CLIPS = []  # file names without .mp4, e.g. ["my_clip"]
STAGES = ["detection", "pitch"]
INCLUDE_VIDEOS = True  # debug_detection.mp4 + minimap.mp4 (about 30 MB per clip)
OUT_ZIP = "/kaggle/working/taktikai_processed.zip"  # Colab: "/content/taktikai_processed.zip"
assert VIDEO_DIR, "Set VIDEO_DIR first"

In [ ]:
# Colab only: mount Google Drive if the clips live there.
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
# Clone once; on re-runs pull instead (an absolute path, so a re-run after %cd does not nest).
from pathlib import Path

REPO_DIR = Path(OUT_ZIP).parent / "taktikai"
if (REPO_DIR / ".git").exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q uv
!uv sync --all-extras
!uv run python -c "import torch; print('CUDA available:', torch.cuda.is_available())"
!git log --oneline -1

In [ ]:
import os
import subprocess
import time

# The Jupyter kernel exports an inline matplotlib backend that the project environment
# does not have; the CLI forces Agg itself, this keeps the subprocess env clean too.
env = {**os.environ, "MPLBACKEND": "Agg"}

for clip in CLIPS:
    for stage in STAGES:
        video = f"{VIDEO_DIR}/{clip}.mp4"
        start = time.time()
        cmd = [
            "uv",
            "run",
            "python",
            "-m",
            "match_tagger.cli",
            "run",
            "--stage",
            stage,
            "--video",
            video,
            "--match",
            clip,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, env=env)
        print(f"{clip} {stage}: exit {result.returncode} in {time.time() - start:.0f} s")
        if result.returncode != 0:
            print(result.stderr[-3000:])
            raise RuntimeError(f"{stage} failed for {clip}")

In [ ]:
import json
import zipfile
from pathlib import Path

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for clip in CLIPS:
        for path in sorted((Path("data/processed") / clip).iterdir()):
            if path.suffix == ".mp4" and not INCLUDE_VIDEOS:
                continue
            zf.write(path, arcname=f"{clip}/{path.name}")
        det = json.loads((Path("data/processed") / clip / "detection_summary.json").read_text())
        keys = ("n_sampled_frames", "n_player_tracks", "ball_frame_share")
        print(clip, {k: det[k] for k in keys})
print(f"Download {OUT_ZIP} ({Path(OUT_ZIP).stat().st_size / 1e6:.0f} MB)")